# 03 — Evaluation & Backbone Ablation

**Purpose:** Load the best checkpoint from training (notebook 02), run full
evaluation on BDD100K val set, and report per-class detection mAP + lane IoU.

**Stage 1 deliverable:** Baseline numbers for the YOLOP-style CSP backbone.
These become the reference for Stage 2 ablations.

**Future (Stage 2):** This notebook will be extended to compare backbone swaps
(e.g., CSPDarknet → EfficientNet, ConvNeXt) one variable at a time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q yacs tqdm opencv-python-headless

import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
import torchvision.transforms as T

from lib.config import cfg
from lib.models import get_net
from lib.core import get_loss, validate
from lib.dataset import BddDataset
from lib.utils.drive_dataset import (
    ensure_local_dataset_from_drive,
    find_raw_bdd_root,
    resolve_bdd_images_100k_dir,
    resolve_bdd_labels_100k_dir,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Default eval target = best row. Set CONFIG='YOLOPv2-focal-only' to
# evaluate the preceding ablation row against it, or 'YOLOP' for the
# honest-YOLOP reference.
CONFIG = 'YOLOPv2-best-row'
yaml_map = {
    'YOLOPv2-paper-no-da': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_paper_no_da.yaml'),
    'YOLOPv2-best-row':   os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_best_row.yaml'),
    'YOLOPv2-focal-only': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_focal_only_ablation.yaml'),
    'YOLOP':              os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolop_vehicle_lane_baseline.yaml'),
}

cfg.defrost()
cfg.merge_from_file(yaml_map[CONFIG])

ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
DATASET_ROOT = ensure_local_dataset_from_drive('bdd100k_vehicle5', ECOCAR_ROOT)
RAW_BDD_ROOT = find_raw_bdd_root(ECOCAR_ROOT)
BDD_IMAGES = resolve_bdd_images_100k_dir(RAW_BDD_ROOT)
BDD_LABELS = resolve_bdd_labels_100k_dir(RAW_BDD_ROOT)
cfg.DATASET.ROOT = DATASET_ROOT
cfg.DATASET.DATAROOT = BDD_IMAGES
cfg.DATASET.LABELROOT = BDD_LABELS
cfg.DATASET.LANEROOT = os.path.join(DATASET_ROOT, 'masks')
# Checkpoint + metrics dirs already correct in each YAML.
cfg.TEST.PLOTS = True
cfg.freeze()

print(f'Eval target  : {CONFIG}')
print(f'NC           : {cfg.MODEL.NC}  (classes: {cfg.MODEL.VEHICLE_CLASSES})')
print(f'Checkpoints  : {cfg.DRIVE.CHECKPOINT_DIR}')
print(f'Dice gain    : {cfg.LOSS.LL_DICE_GAIN}')

In [ ]:
# ── Load model + best checkpoint ──
# get_net reads cfg.MODEL.NC and names — do NOT override them.
model = get_net(cfg).to(device)
model.gr = 1.0

ckpt_path = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth')
assert os.path.exists(ckpt_path), f'No checkpoint found at {ckpt_path}'
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')
print(f'  nc={model.nc}  names={model.names}')
print(f'  best_map={ckpt.get("best_map", "N/A")}, best_ll_iou={ckpt.get("best_ll_iou", "N/A")}')

In [ ]:
# ── Build val dataset ──
# YOLOPv2 paper §3: test letterbox 640×384. Pass an explicit (H, W)
# tuple so BddDataset uses letterbox(auto=False) instead of the
# stride-rounded auto=True path.
val_wh = tuple(getattr(cfg.TEST, 'IMAGE_SIZE', [640, 640]))   # (W, H)
val_size = (int(val_wh[1]), int(val_wh[0]))                   # (H, W)

val_dataset = BddDataset(cfg, is_train=False, inputsize=val_size, transform=T.ToTensor())
val_loader = DataLoader(
    val_dataset, batch_size=cfg.TEST.BATCH_SIZE_PER_GPU,
    shuffle=False, num_workers=4,
    pin_memory=True, collate_fn=val_dataset.collate_fn,
)
sample = val_dataset[0][0].shape
print(f'Val: {len(val_dataset)} samples, letterbox C×H×W = {tuple(sample)}')

In [ ]:
# ── Run evaluation ──
import logging
logger = logging.getLogger('eval')
logger.setLevel(logging.INFO)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())

criterion = get_loss(cfg, device)
output_dir = cfg.DRIVE.METRICS_DIR

ll_seg_result, det_result, val_loss, maps, times = validate(
    epoch=ckpt['epoch'], config=cfg, val_loader=val_loader,
    val_dataset=val_dataset, model=model, criterion=criterion,
    output_dir=output_dir, tb_log_dir='', writer_dict=None,
    logger=logger, device=device
)

ll_acc, ll_iou, ll_miou = ll_seg_result
mp, mr, map50, map_all = det_result

print('\n' + '='*60)
print('BASELINE RESULTS (YOLOP CSP Backbone)')
print('='*60)
print(f'Detection:')
print(f'  Precision:   {mp:.4f}')
print(f'  Recall:      {mr:.4f}')
print(f'  mAP@0.5:     {map50:.4f}')
print(f'  mAP@0.5:0.95:{map_all:.4f}')
print(f'Lane Segmentation:')
print(f'  Accuracy:    {ll_acc:.4f}')
print(f'  IoU:         {ll_iou:.4f}')
print(f'  mIoU:        {ll_miou:.4f}')
print(f'Timing:')
print(f'  Inference:   {times[0]*1000:.1f} ms/img')
print(f'  NMS:         {times[1]*1000:.1f} ms/img')
print(f'Val loss:      {val_loss:.4f}')
print('='*60)

In [ ]:
# ── Save results for comparison ──
import json

results = {
    'backbone': 'CSP (YOLOP baseline)',
    'epoch': ckpt['epoch'],
    'detection': {'precision': float(mp), 'recall': float(mr),
                  'mAP50': float(map50), 'mAP': float(map_all)},
    'lane': {'accuracy': float(ll_acc), 'IoU': float(ll_iou), 'mIoU': float(ll_miou)},
    'val_loss': float(val_loss),
    'inference_ms': float(times[0]*1000),
}

results_path = os.path.join(cfg.DRIVE.METRICS_DIR, 'baseline_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {results_path}')